# Rego - regulation-triggered retraining (CT)

Runs on Kaggle (P100 GPU). Trains an XGBoost classifier that **cannot use the features active
regulation rules prohibit**, then writes a model bundle for the CI gates:

- `model.pkl` - the constrained model
- `profile.json` - one `<feature>_weight` per feature (normalized importance; prohibited features are exactly 0)
- `evaluation.json` - held-out predictions for the fairness and regression gates. `baseline_y_pred` comes from an
  unconstrained model trained on all features, so the regression gate measures the accuracy cost of compliance.

The constraint is applied by excluding prohibited columns (a hard guarantee of zero weight), not by a soft
penalty: gradient-boosted trees have no per-feature weight to penalize.


In [ ]:
import json

# Filled in by the pusher (src/pipeline/ct/kaggle_runner.py) before the run.
CONFIG = json.loads(r'''__CT_CONFIG__''')
REGULATION_VERSION = CONFIG["regulation_version"]
PROHIBITED = set(CONFIG["prohibited_features"])
TARGET_COLUMN = CONFIG.get("target_column", "approved")
SENSITIVE_COLUMN = CONFIG.get("sensitive_column", "gender")
print("Regulation version:", REGULATION_VERSION)
print("Prohibited features:", sorted(PROHIBITED))


In [ ]:
import glob
import pandas as pd

csv_files = sorted(glob.glob("/kaggle/input/*/*.csv"))
assert csv_files, "No training CSV found in the attached Kaggle dataset."
df = pd.read_csv(csv_files[0])
assert TARGET_COLUMN in df.columns, f"Missing target column {TARGET_COLUMN!r}"
assert SENSITIVE_COLUMN in df.columns, f"Missing sensitive column {SENSITIVE_COLUMN!r}"

numeric = df.select_dtypes("number").columns
all_features = [c for c in numeric if c not in (TARGET_COLUMN, SENSITIVE_COLUMN)]
allowed = [c for c in all_features if c not in PROHIBITED]
print(f"{len(all_features)} candidate features, {len(allowed)} allowed")


In [ ]:
# Privacy guard: refuse to train on personal data.
# Training data must be anonymized before it reaches this notebook: no direct
# identifiers, and any ID column holds hashed IDs only (64 hex characters).
PII_COLUMNS = {
    "name", "first_name", "last_name", "full_name", "email", "phone", "mobile",
    "pan", "aadhaar", "aadhar", "address", "dob", "date_of_birth", "ssn",
}
ID_COLUMNS = {"applicant_id", "customer_id", "id"}

found = sorted(c for c in df.columns if c.lower() in PII_COLUMNS)
assert not found, f"Training data has personal data columns {found}. Anonymize it first."
for column in (c for c in df.columns if c.lower() in ID_COLUMNS):
    hashed = df[column].astype(str).str.fullmatch(r"[0-9a-f]{64}").all()
    assert hashed, f"{column!r} must contain hashed IDs only (64-character hex)."


In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET_COLUMN])
y_train, y_test = train[TARGET_COLUMN], test[TARGET_COLUMN]


In [ ]:
from xgboost import XGBClassifier

def fit(columns):
    model = XGBClassifier(n_estimators=200, max_depth=4, tree_method="hist", device="cuda", random_state=42)
    model.fit(train[columns], y_train)
    return model

baseline_model = fit(all_features)   # unconstrained reference
constrained_model = fit(allowed)     # never sees prohibited columns

y_pred = constrained_model.predict(test[allowed]).astype(int).tolist()
baseline_pred = baseline_model.predict(test[all_features]).astype(int).tolist()


In [ ]:
import pickle
from pathlib import Path

out = Path("/kaggle/working")
weights = {f"{c}_weight": 0.0 for c in all_features}
for column, importance in zip(allowed, constrained_model.feature_importances_):
    weights[f"{column}_weight"] = float(importance)
assert all(weights[f"{c}_weight"] == 0.0 for c in PROHIBITED if f"{c}_weight" in weights)

(out / "model.pkl").write_bytes(pickle.dumps(constrained_model))
(out / "profile.json").write_text(json.dumps({"regulation_version": REGULATION_VERSION, "weights": weights}))
(out / "evaluation.json").write_text(json.dumps({
    "y_true": y_test.astype(int).tolist(),
    "y_pred": y_pred,
    "baseline_y_pred": baseline_pred,
    "groups": test[SENSITIVE_COLUMN].astype(str).tolist(),
}))
print(json.dumps(weights, indent=2))
